In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import os, glob, warnings, numpy as np, pandas as pd, xarray as xr
warnings.filterwarnings('ignore')

O   = rf'{DATA_ROOT}/new_climate/_proj_main'
M2C = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid2'
OUT = os.path.join(O, 'tables'); os.makedirs(OUT, exist_ok=True)

BASE_Y  = (2002, 2020)          # reference for Δ: ERA5 observations
QDM_REF = (2002, 2014)          # QDM calibration period (fixed; used only for the GOOD mask)
SLICES  = {'2021-2040':(2021,2040), '2041-2060':(2041,2060),
           '2061-2080':(2061,2080), '2081-2100':(2081,2100)}
MODELS  = ['ACCESS','ECEARTH']
SSPS    = ['ssp126','ssp585']
COVER_MIN = 0.9

# five cumulative thresholds + six mutually exclusive bins (identical to M2 Table 1)
EDGES = {'wave':[0,1.25,2.5,4.0,6.0,9.0,np.inf], 'wind':[0,8.0,10.8,13.9,17.2,20.8,np.inf]}
CUTS  = {'wave':[1.25,2.5,4.0,6.0,9.0],          'wind':[8.0,10.8,13.9,17.2,20.8]}
def bin_names(k):
    e=EDGES[k]; return [f'D_{e[i]:g}_{"inf" if np.isinf(e[i+1]) else f"{e[i+1]:g}"}' for i in range(6)]
BINS = {k:bin_names(k) for k in EDGES}
# bin → M2 coefficient name
COEF_KEY = {
 'D_1.25_2.5':'Wv_1p25_2p5','D_2.5_4':'Wv_2p5_4','D_4_6':'Wv_4_6','D_6_9':'Wv_6_9','D_9_inf':'Wv_9_p',
 'D_8_10.8':'Wd_8p0_10p8','D_10.8_13.9':'Wd_10p8_13p9','D_13.9_17.2':'Wd_13p9_17p2',
 'D_17.2_20.8':'Wd_17p2_20p8','D_20.8_inf':'Wd_20p8_p'}

REG = {
 'NW_Europe':[(36,45,-15,-6),(45,63,-15,-6),(43,63,-6,0),(48,63,0,13)],
 'NW_Atlantic':[(18,31,-98,-60),(31,42,-82,-60),(42,50,-67,-60),(45,55,-60,-46)],
 'Med_Black':[(30,43,-6,0),(30,46,0,19),(30,46,19,26.5),(30,41,26.5,37),(41,47.5,26.5,42)],
 'Asia':[(-10,20,95,130),(20,46,117,146)],
 'Indian_Ocean_N':[(5,30,50,78),(5,26,78,95)]}
REGS = list(REG)

# cell list and region membership (obs cell order is canonical; all caches share it)
cells = xr.open_dataarray(os.path.join(O,'obs_wave.nc')).cell.values
def reg_of(g):
    la,lo = map(float,g.split('_'))
    for r,bs in REG.items():
        for S,N,W,E in bs:
            if S<=la<N and W<=lo<E: return r
    return None
RG = np.array([reg_of(g) for g in cells])

# GOOD: coverage ≥90% in the calibration period for all four sets (2 models × wind/wave) (Baltic and Black Sea therefore dropped entirely)
_t = pd.to_datetime(xr.open_dataarray(os.path.join(O,'cm_wave_ACCESS_his.nc')).time.values)
_k = (_t.year>=QDM_REF[0]) & (_t.year<=QDM_REF[1])
GOOD = np.ones(len(cells), bool)
for m in MODELS:
    for k in ('wave','wind'):
        GOOD &= np.isfinite(xr.open_dataarray(os.path.join(O,f'cm_{k}_{m}_his.nc')).load().values[_k]).mean(0) >= COVER_MIN
RMASK = {r: (RG==r) & GOOD for r in REGS}
print('usable cells', int(GOOD.sum()), '/', len(cells))
print(pd.Series({r:int(v.sum()) for r,v in RMASK.items()}).to_string())


In [ ]:
def monthly_metrics(arr, times, kind):
    """(T,NCELL) → all metrics by region × calendar month; returns DataFrame[region,cmonth,metric,value]"""
    ym  = times.year.values*100 + times.month.values
    u   = np.unique(ym); idx = np.searchsorted(ym, u)
    ok  = np.isfinite(arr); e = EDGES[kind]
    acc = {}
    acc['n_days'] = np.add.reduceat(ok.astype(np.float32), idx, 0)
    for c in CUTS[kind]:
        acc[f'N_{c:g}'] = np.add.reduceat((ok&(arr>=c)).astype(np.float32), idx, 0)
    for i,nm in enumerate(BINS[kind]):
        acc[nm] = np.add.reduceat((ok&(arr>=e[i])&(arr<e[i+1])).astype(np.float32), idx, 0)
    rows=[]
    mo = u % 100
    for r in REGS:
        k = RMASK[r]
        for nm,v in acc.items():
            s = pd.DataFrame({'cmonth':mo,'v':v[:,k].mean(1)}).groupby('cmonth').v.mean()
            rows.append(pd.DataFrame({'region':r,'cmonth':s.index,'metric':nm,'value':s.values}))
    return pd.concat(rows, ignore_index=True)

def from_m2_cache(kind,y0,y1):
    fr=[]
    for f in glob.glob(os.path.join(M2C,f'*__dmax_{kind}.parquet')):
        d=pd.read_parquet(f); d['date']=pd.to_datetime(d['date'])
        fr.append(d[(d.date.dt.year>=y0)&(d.date.dt.year<=y1)])
    d=pd.concat(fr,ignore_index=True).drop_duplicates(['grid_id','date'])
    p=d.pivot(index='date',columns='grid_id',values='dmax').sort_index().reindex(columns=cells)
    return p.values.astype(np.float32), pd.DatetimeIndex(p.index)

BASE={}
for kind in ('wave','wind'):
    o = xr.open_dataarray(os.path.join(O,f'obs_{kind}.nc')).load()      # 2002-2014
    a,t = from_m2_cache(kind, 2015, BASE_Y[1])                          # 2015-2020, taken from cache at no cost
    arr = np.concatenate([o.values, a])
    tim = pd.DatetimeIndex(np.concatenate([pd.to_datetime(o.time.values), np.asarray(t)]))
    s = np.argsort(tim.values); arr, tim = arr[s], tim[s]
    exp = pd.date_range(f'{BASE_Y[0]}-01-01', f'{BASE_Y[1]}-12-31', freq='D')
    assert len(tim)==len(exp) and (tim==exp).all(), f'{kind} baseline time axis incomplete'
    BASE[kind] = monthly_metrics(arr, tim, kind).rename(columns={'value':'base'})
    print(f'baseline {kind}: ERA5 {BASE_Y[0]}-{BASE_Y[1]}  {arr.shape}')


In [ ]:
A=[]
for kind in ('wave','wind'):
    for m in MODELS:
        for ssp in SSPS:
            q = xr.open_dataarray(os.path.join(O,f'qdm_{kind}_{m}_{ssp}.nc')).load()
            t = pd.to_datetime(q.time.values); v = q.values
            for sl,(y0,y1) in SLICES.items():
                k = (t.year>=y0)&(t.year<=y1)
                f = monthly_metrics(v[k], t[k], kind).rename(columns={'value':'future'})
                d = f.merge(BASE[kind], on=['region','cmonth','metric'])
                d['delta'] = d.future - d.base
                d.insert(0,'variable',kind); d.insert(0,'slice',sl)
                d.insert(0,'scenario',ssp); d.insert(0,'model',m)
                A.append(d)
            del q,v
            print(f'  {kind} {m} {ssp} done', flush=True)
TA = pd.concat(A, ignore_index=True)
TA.to_parquet(os.path.join(OUT,'tableA_hazard_monthly.parquet'), index=False)
print('\nTable A', TA.shape, '→ tableA_hazard_monthly.parquet')

# Self-check: sum of six bins == n_days
chk = (TA[TA.metric.isin(BINS['wave'])].groupby(['model','scenario','slice','region','cmonth']).future.sum()
       - TA[(TA.metric=='n_days')&(TA.variable=='wave')].set_index(
           ['model','scenario','slice','region','cmonth']).future)
print('max bin-closure error', float(chk.abs().max()), '(should be 0)')


In [ ]:
S = pd.read_csv(os.path.join(M2C,'M2_results_summary.csv'))
BETA = {b:{x.label.split('/')[1]: np.log(x.IRR) for _,x in
           S[(S.block==b)&S.label.str.startswith('A_it/W')].iterrows()} for b in ('RQ1','RQ1-joint')}

pg = pd.read_parquet(os.path.join(M2C,'panel_global.parquet'),
                     columns=['region','grid_id','year_month','A_it','E_it'])
pg = pg[pg.grid_id.isin(set(cells[GOOD]))]
pg['cmonth'] = pd.to_datetime(pg.year_month).dt.month
W = pg.groupby(['region','cmonth'])[['A_it','E_it']].sum().reset_index().rename(
        columns={'A_it':'w_acc','E_it':'w_trf'})

d = TA[TA.metric.isin(COEF_KEY)].copy()
d['coef_joint'] = d.metric.map(lambda x: BETA['RQ1-joint'].get(COEF_KEY[x],0.0))
d['coef_rq1']   = d.metric.map(lambda x: BETA['RQ1'].get(COEF_KEY[x],0.0))
d['lr_joint'] = d.delta*d.coef_joint
d['lr_rq1']   = d.delta*d.coef_rq1
g = d.groupby(['model','scenario','slice','region','cmonth','variable'])[['lr_joint','lr_rq1']].sum().reset_index()
p = g.pivot_table(index=['model','scenario','slice','region','cmonth'],
                  columns='variable', values=['lr_joint','lr_rq1']).reset_index()
p.columns = ['model','scenario','slice','region','cmonth',
             'LR_joint_wave','LR_joint_wind','LR_rq1_wave','LR_rq1_wind']
p['LR_joint'] = p.LR_joint_wave + p.LR_joint_wind          # ★ main specification
p['RR_joint']    = np.exp(p.LR_joint)
p['RR_rq1_wave'] = np.exp(p.LR_rq1_wave)                   # wave-only total effect
p['RR_rq1_wind'] = np.exp(p.LR_rq1_wind)                   # wind-only total effect
TB = p.merge(W, on=['region','cmonth'], how='left')
TB.to_parquet(os.path.join(OUT,'tableB_risk_monthly.parquet'), index=False)
print('Table B', TB.shape, '→ tableB_risk_monthly.parquet')


In [ ]:
SEAS = {'DJF':[12,1,2],'MAM':[3,4,5],'JJA':[6,7,8],'SON':[9,10,11]}
SMAP = {m:s for s,ms in SEAS.items() for m in ms}
MIN_BASE = 0.5      # annual baseline <0.5 days/yr → no percentage reported

# ── A1: annual hazard (region × metric × period)──
a = TA.groupby(['model','scenario','slice','region','variable','metric'],
               as_index=False)[['base','future','delta']].sum()
a['pct'] = np.where(a.base>=MIN_BASE, a.delta/a.base*100, np.nan)
a.to_csv(os.path.join(OUT,'A1_hazard_annual.csv'), index=False)

# ── A2: seasonal hazard ──
s = TA.assign(season=TA.cmonth.map(SMAP)).groupby(
    ['model','scenario','slice','region','variable','metric','season'],
    as_index=False)[['base','future','delta']].sum()
s['pct'] = np.where(s.base>=MIN_BASE/4, s.delta/s.base*100, np.nan)
s.to_csv(os.path.join(OUT,'A2_hazard_seasonal.csv'), index=False)

# ── B1/B2: risk by region (annual / seasonal), monthly RR combined with accident weights ──
def collapse(df, keys, rr='RR_joint', w='w_acc'):
    return (df.assign(_n=df[w]*df[rr], _d=df[w]).groupby(keys, as_index=False)[['_n','_d']].sum()
              .assign(**{f'{rr}_agg':lambda x:x._n/x._d})
              .drop(columns=['_n','_d']))
out=[]
for rr in ('RR_joint','RR_rq1_wave','RR_rq1_wind'):
    for wname in ('w_acc','w_trf'):
        for lab,keys,frame in [
            ('ANN',['model','scenario','slice','region'],TB),
            ('SEA',['model','scenario','slice','region','season'],TB.assign(season=TB.cmonth.map(SMAP)))]:
            c = collapse(frame, keys, rr, wname).rename(columns={f'{rr}_agg':'RR'})
            c['pct']=(c.RR-1)*100; c['coef']=rr; c['weight']=wname; c['level']=lab
            out.append(c)
RISK_REG = pd.concat(out, ignore_index=True)
RISK_REG.to_csv(os.path.join(OUT,'B1_risk_region.csv'), index=False)

# ── B3: five-region total + MWRI index (reference = 100)──
out=[]
for rr in ('RR_joint','RR_rq1_wave','RR_rq1_wind'):
    for wname in ('w_acc','w_trf'):
        for lab,keys,frame in [
            ('ANN',['model','scenario','slice'],TB),
            ('SEA',['model','scenario','slice','season'],TB.assign(season=TB.cmonth.map(SMAP)))]:
            c = collapse(frame, keys, rr, wname).rename(columns={f'{rr}_agg':'RR'})
            c['pct']=(c.RR-1)*100; c['MWRI']=c.RR*100
            c['coef']=rr; c['weight']=wname; c['level']=lab
            out.append(c)
RISK_TOT = pd.concat(out, ignore_index=True)
RISK_TOT.to_csv(os.path.join(OUT,'B2_risk_total.csv'), index=False)

# ── B4: contribution of each region to the total (percentage points)──
base_tot = TB.groupby(['model','scenario','slice']).w_acc.sum().rename('W')
contrib = (TB.assign(x=TB.w_acc*(TB.RR_joint-1))
             .groupby(['model','scenario','slice','region']).x.sum().reset_index()
             .merge(base_tot, on=['model','scenario','slice']))
contrib['pp'] = contrib.x/contrib.W*100
contrib.drop(columns=['x','W']).to_csv(os.path.join(OUT,'B3_region_contribution.csv'), index=False)

# ── Preview ──
pd.set_option('display.width',240)
print('\n=== MWRI index (reference 2002-2020 = 100, accident-weighted, joint coefficients)===')
t=RISK_TOT[(RISK_TOT.level=='ANN')&(RISK_TOT.coef=='RR_joint')&(RISK_TOT.weight=='w_acc')]
print(t.pivot_table(index='slice',columns=['scenario','model'],values='MWRI').round(1).to_string())
print('\n=== annual hazard: Δ days/yr (SSP5-8.5, 2081-2100)===')
q=a[(a.scenario=='ssp585')&(a['slice']=='2081-2100')&a.metric.str.startswith('N_')]
print(q.pivot_table(index=['variable','metric'],columns=['region','model'],values='delta').round(1).to_string())
print('\nOutput folder:', OUT)


In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.rcParams.update({'font.sans-serif':['Arial', 'DejaVu Sans'],
                     'axes.unicode_minus':False})

TB = rf'{DATA_ROOT}/new_climate/_proj_main/tables'
TA = pd.read_parquet(os.path.join(TB,'tableA_hazard_monthly.parquet'))

REGS  = ['NW_Europe','NW_Atlantic','Med_Black','Asia','Indian_Ocean_N']
RLAB  = {'NW_Europe':'Northwest Europe','NW_Atlantic':'Northwest Atlantic',
         'Med_Black':'Mediterranean','Asia':'East & SE Asia','Indian_Ocean_N':'North Indian Ocean'}
SLICES= ['2021-2040','2041-2060','2061-2080','2081-2100']
SEAS  = ['DJF','MAM','JJA','SON']
SMAP  = {12:'DJF',1:'DJF',2:'DJF',3:'MAM',4:'MAM',5:'MAM',
         6:'JJA',7:'JJA',8:'JJA',9:'SON',10:'SON',11:'SON'}
MLAB  = {'ACCESS':'ACCESS-CM2','ECEARTH':'EC-Earth3'}
SLAB  = {'ssp126':'SSP1-2.6','ssp585':'SSP5-8.5'}
SC    = {'ssp126':'#2b7bba','ssp585':'#c8452b'}
DASH  = {'ACCESS':'-','ECEARTH':(0,(5,2))}
MK    = {'ACCESS':'o','ECEARTH':'s'}
UNIT  = {'wave':'m','wind':'m s$^{-1}$'}
SYM   = {'wave':'$H_s$','wind':'$U_{10}$'}
MIN_BASE = 0.5

ANN = TA.groupby(['model','scenario','slice','region','variable','metric'],
                 as_index=False)[['base','future','delta']].sum()
ANN['pct'] = np.where(ANN.base>=MIN_BASE, ANN.delta/ANN.base*100, np.nan)

SEA = (TA.assign(season=TA.cmonth.map(SMAP))
         .groupby(['model','scenario','slice','region','variable','metric','season'],
                  as_index=False)[['base','future','delta']].sum())
SEA['pct'] = np.where(SEA.base>=MIN_BASE/4, SEA.delta/SEA.base*100, np.nan)
print('ANN', ANN.shape, ' SEA', SEA.shape)


In [ ]:
VALUE = 'delta'          # 'delta' = days/yr;'pct' = % (comparable across regions, shared y axis)
YLAB  = {'delta':'Δ days yr$^{-1}$','pct':'Change (%)'}[VALUE]

def fig_slices(variable, cuts):
    d  = ANN[(ANN.variable==variable) & (ANN.metric.isin([f'N_{c:g}' for c in cuts]))]
    nr, nc = len(REGS), len(cuts)
    fig, axes = plt.subplots(nr, nc, figsize=(3.6*nc, 2.5*nr), sharex=True,
                             sharey=('col' if VALUE=='pct' else False), dpi=115)
    axes = np.atleast_2d(axes)
    if nc == 1: axes = axes.reshape(-1, 1)
    x = np.arange(len(SLICES))
    for i, r in enumerate(REGS):
        for j, c in enumerate(cuts):
            ax  = axes[i, j]
            sub = d[(d.region==r) & (d.metric==f'N_{c:g}')]
            base = sub.base.mean()
            for m in MLAB:
                for ssp in SLAB:
                    s = (sub[(sub.model==m)&(sub.scenario==ssp)]
                         .set_index('slice').reindex(SLICES))
                    ax.plot(x, s[VALUE].values, ls=DASH[m], color=SC[ssp],
                            marker=MK[m], ms=5, lw=1.9, zorder=3)
            ax.axhline(0, color='#4a4a4a', lw=1, zorder=1)
            ax.grid(alpha=.22, lw=.5)
            ax.set_xticks(x)
            ax.set_xticklabels([s[2:4]+'–'+s[-2:] for s in SLICES], fontsize=8.5)
            if i == 0: ax.set_title(f'{SYM[variable]} ≥ {c:g} {UNIT[variable]}', fontsize=11.5, pad=7)
            if j == 0: ax.set_ylabel(f'{RLAB[r]}\n{YLAB}', fontsize=9)
            ax.text(.03,.06,f'base {base:.1f} d/yr', transform=ax.transAxes,
                    fontsize=7.5, color='#6b7178')
            for sp in ('top','right'): ax.spines[sp].set_visible(False)
    hd = ([Line2D([],[],color=SC[s],lw=2.2,label=SLAB[s]) for s in SLAB] +
          [Line2D([],[],color='#1a1a1a',ls=DASH[m],marker=MK[m],ms=5,lw=1.7,label=MLAB[m])
           for m in MLAB])
    fig.legend(handles=hd, loc='upper center', ncol=4, frameon=False, fontsize=11,
               bbox_to_anchor=(.5,1.005))
    fig.suptitle('Change in exceedance days — '
                 f'{"significant wave height" if variable=="wave" else "10-m wind speed"}'
                 '   (vs ERA5 2002–2020)', y=1.035, fontsize=13.5)
    plt.tight_layout(); plt.show()

fig_slices('wave', [2.5, 4.0, 6.0])
fig_slices('wind', [10.8, 13.9, 17.2])


In [ ]:
CMAP_SL = plt.cm.YlOrRd(np.linspace(.35,.92,len(SLICES)))    # period: light = near term, dark = end of century

def fig_season(variable, cut, value='delta'):
    d = SEA[(SEA.variable==variable) & (SEA.metric==f'N_{cut:g}')]
    ylab = {'delta':'Δ days per season','pct':'Change (%)'}[value]
    fig, axes = plt.subplots(len(REGS), 2, figsize=(9.6, 2.5*len(REGS)),
                             sharex=True, sharey='row', dpi=115)
    axes = np.atleast_2d(axes)
    x = np.arange(4)
    for i, r in enumerate(REGS):
        for j, ssp in enumerate(['ssp126','ssp585']):
            ax = axes[i, j]
            for k, sl in enumerate(SLICES):
                for m in MLAB:
                    s = (d[(d.region==r)&(d.scenario==ssp)&(d['slice']==sl)&(d.model==m)]
                         .set_index('season').reindex(SEAS))
                    ax.plot(x, s[value].values, ls=DASH[m], color=CMAP_SL[k],
                            lw=2.0, marker=MK[m], ms=4.5, zorder=3)
            ax.axhline(0, color='#4a4a4a', lw=1, zorder=1)
            ax.set_xticks(x); ax.set_xticklabels(SEAS, fontsize=9.5)
            ax.grid(alpha=.22, lw=.5)
            if i == 0: ax.set_title(SLAB[ssp], fontsize=12, pad=7)
            if j == 0: ax.set_ylabel(f'{RLAB[r]}\n{ylab}', fontsize=9)
            for sp in ('top','right'): ax.spines[sp].set_visible(False)
    hd = ([Line2D([],[],color=CMAP_SL[k],lw=2.2,label=sl) for k,sl in enumerate(SLICES)] +
          [Line2D([],[],color='#1a1a1a',ls=DASH[m],marker=MK[m],ms=4.5,lw=1.7,label=MLAB[m])
           for m in MLAB])
    fig.legend(handles=hd, loc='upper center', ncol=6, frameon=False, fontsize=10.5,
               bbox_to_anchor=(.5,1.005))
    fig.suptitle(f'Seasonal change in days with {SYM[variable]} ≥ {cut:g} {UNIT[variable]}'
                 '   (vs ERA5 2002–2020)', y=1.032, fontsize=13.5)
    plt.tight_layout(); plt.show()

fig_season('wave', 4.0)
fig_season('wind', 13.9)


In [ ]:
VALUE = 'delta'          # 'delta' = days/yr;'pct' = %
YLAB  = {'delta':'Δ days yr$^{-1}$','pct':'Change (%)'}[VALUE]

def fig_slices(variable, cuts):
    d = ANN[(ANN.variable==variable) & (ANN.metric.isin([f'N_{c:g}' for c in cuts]))]
    nr, nc = len(REGS), len(cuts)
    fig, axes = plt.subplots(nr, nc, figsize=(3.6*nc, 2.5*nr), sharex=True,
                             sharey=('col' if VALUE=='pct' else False), dpi=115)
    x = np.arange(len(SLICES))
    for i, r in enumerate(REGS):
        for j, c in enumerate(cuts):
            ax = axes[i, j]
            sub = d[(d.region==r) & (d.metric==f'N_{c:g}')]
            base = sub.base.mean()
            for m in MLAB:
                for ssp in SLAB:
                    s = sub[(sub.model==m)&(sub.scenario==ssp)].set_index('slice').reindex(SLICES)
                    ax.plot(x, s[VALUE].values, DASH[m], color=SC[ssp], marker=MK[m],
                            ms=5, lw=1.9, zorder=3)
            ax.axhline(0, color='#4a4a4a', lw=1, zorder=1)
            ax.grid(alpha=.22, lw=.5)
            ax.set_xticks(x); ax.set_xticklabels([s[2:4]+'–'+s[-2:] for s in SLICES], fontsize=8.5)
            if i == 0: ax.set_title(f'{SYM[variable]} ≥ {c:g} {UNIT[variable]}', fontsize=11.5, pad=7)
            if j == 0: ax.set_ylabel(f'{RLAB[r]}\n{YLAB}', fontsize=9)
            ax.text(.03,.06,f'base {base:.1f} d/yr',transform=ax.transAxes,
                    fontsize=7.5,color='#6b7178')
            for sp in ('top','right'): ax.spines[sp].set_visible(False)
    hd = ([Line2D([],[],color=SC[s],lw=2.2,label=SLAB[s]) for s in SLAB] +
          [Line2D([],[],color='#1a1a1a',ls=DASH[m],marker=MK[m],ms=5,lw=1.7,label=MLAB[m]) for m in MLAB])
    fig.legend(handles=hd, loc='upper center', ncol=4, frameon=False, fontsize=11,
               bbox_to_anchor=(.5,1.005))
    fig.suptitle(f'Change in exceedance days — {"significant wave height" if variable=="wave" else "10-m wind speed"}'
                 f'   (vs ERA5 2002–2020)', y=1.035, fontsize=13.5)
    plt.tight_layout(); plt.show()

fig_slices('wave', [2.5, 4.0, 6.0])
fig_slices('wind', [10.8, 13.9, 17.2])


In [ ]:
CMAP_SL = plt.cm.YlOrRd(np.linspace(.35,.92,len(SLICES)))   # periods light to dark

def fig_season(variable, cut, value='delta'):
    d = SEA[(SEA.variable==variable) & (SEA.metric==f'N_{cut:g}')]
    ylab = {'delta':'Δ days per season','pct':'Change (%)'}[value]
    fig, axes = plt.subplots(len(REGS), 2, figsize=(9.6, 2.5*len(REGS)),
                             sharex=True, sharey='row', dpi=115)
    x = np.arange(4)
    for i, r in enumerate(REGS):
        for j, ssp in enumerate(['ssp126','ssp585']):
            ax = axes[i, j]
            for k, sl in enumerate(SLICES):
                for m in MLAB:
                    s = (d[(d.region==r)&(d.scenario==ssp)&(d['slice']==sl)&(d.model==m)]
                         .set_index('season').reindex(SEAS))
                    ax.plot(x, s[value].values, DASH[m], color=CMAP_SL[k], lw=2.0,
                            marker=MK[m], ms=4.5, zorder=3)
            ax.axhline(0, color='#4a4a4a', lw=1, zorder=1)
            ax.set_xticks(x); ax.set_xticklabels(SEAS, fontsize=9.5)
            ax.grid(alpha=.22, lw=.5)
            if i == 0: ax.set_title(SLAB[ssp], fontsize=12, pad=7)
            if j == 0: ax.set_ylabel(f'{RLAB[r]}\n{ylab}', fontsize=9)
            for sp in ('top','right'): ax.spines[sp].set_visible(False)
    hd = ([Line2D([],[],color=CMAP_SL[k],lw=2.2,label=sl) for k,sl in enumerate(SLICES)] +
          [Line2D([],[],color='#1a1a1a',ls=DASH[m],marker=MK[m],ms=4.5,lw=1.7,label=MLAB[m]) for m in MLAB])
    fig.legend(handles=hd, loc='upper center', ncol=6, frameon=False, fontsize=10.5,
               bbox_to_anchor=(.5,1.005))
    fig.suptitle(f'Seasonal change in days with {SYM[variable]} ≥ {cut:g} {UNIT[variable]}'
                 f'   (vs ERA5 2002–2020)', y=1.032, fontsize=13.5)
    plt.tight_layout(); plt.show()

fig_season('wave', 4.0)
fig_season('wind', 13.9)
